## 0. Setup — datos de v01 y v02

In [7]:
# --- SETUP: VARIABLES DE SEMANAS ANTERIORES ---

!pip install vectorbt -q
import vectorbt as vbt
import yfinance as yf
import pandas as pd
import numpy as np

# ── Del v02: precios de cotizadas ──────────────────────────
tickers = ['DAL', 'UAL', 'AAL']
nombres = {'DAL': 'Delta', 'UAL': 'United', 'AAL': 'American'}
precios = yf.download(tickers, period='2y', auto_adjust=True)['Close']
precios.columns = [nombres[t] for t in precios.columns]
precios = precios.dropna()

# ── Del v01: datos PYMEs y backtest ────────────────────────
datos_temporales = {
    'nombre': ['Ferretería García']*3 + ['Bar El Rincón']*3 +
              ['Peluquería Ana']*3 + ['Tienda Moda Sol']*3 +
              ['Gestoría Pérez']*3 + ['Clínica Dental Martínez']*3 +
              ['Academia Idiomas Sol']*3 + ['Clínica Dental García']*3 +
              ['Academia Idiomas English House']*3,
    'año': [2022, 2023, 2024] * 9,
    'facturacion': [
        295000, 308000, 320000, 165000, 172000, 180000,
        85000,  90000,  95000,  195000, 202000, 210000,
        125000, 133000, 140000, 255000, 268000, 280000,
        145000, 152000, 160000, 270000, 283000, 295000,
        138000, 146000, 155000,
    ],
    'margen_bruto_pct': [
        32.5, 33.1, 34.4, 46.8, 47.0, 47.2,
        59.2, 59.5, 60.0, 32.5, 32.9, 33.3,
        66.5, 67.2, 67.9, 59.0, 59.5, 60.0,
        69.5, 69.7, 70.0, 59.5, 59.8, 60.0,
        49.5, 49.7, 50.0,
    ]
}
df_temporal = pd.DataFrame(datos_temporales)
df_temporal['fecha'] = pd.to_datetime(df_temporal['año'].astype(str) + '-01-01')
df_temporal = df_temporal.set_index('fecha')
crecimiento = df_temporal.groupby('nombre')['facturacion'].pct_change() * 100
df_temporal['crecimiento_pct'] = crecimiento.round(1)

df_2022 = df_temporal[df_temporal['año'] == 2022].copy()
df_2022 = df_2022.set_index('nombre')
df_2022['pct_margen'] = df_2022['margen_bruto_pct'].rank(pct=True).round(2)
df_2022['pct_crecimiento'] = 0.5
df_2022['score_2022'] = (df_2022['pct_margen'] * 0.60 +
                         df_2022['pct_crecimiento'] * 0.40).round(3)
df_2022['cuartil'] = pd.cut(df_2022['score_2022'],
                             bins=[0, 0.33, 0.66, 1.0],
                             labels=['LOW', 'MID', 'TOP'])

df_2024 = df_temporal[df_temporal['año'] == 2024].set_index('nombre')
df_2022_base = df_temporal[df_temporal['año'] == 2022].set_index('nombre')
inversion_por_cuartil = {'TOP': 10000, 'MID': 5000, 'LOW': 2000}
resultados = []
for empresa in df_2022.index:
    cuartil = df_2022.loc[empresa, 'cuartil']
    inversion = inversion_por_cuartil[str(cuartil)]
    fact_2022 = df_2022_base.loc[empresa, 'facturacion']
    fact_2024 = df_2024.loc[empresa, 'facturacion']
    crecimiento_real = (fact_2024 - fact_2022) / fact_2022
    valor_final = round(inversion * (1 + crecimiento_real), 0)
    ganancia = round(valor_final - inversion, 0)
    resultados.append({
        'empresa': empresa,
        'cuartil_2022': str(cuartil),
        'score_2022': df_2022.loc[empresa, 'score_2022'],
        'inversion': inversion,
        'crecimiento_real_pct': round(crecimiento_real * 100, 1),
        'valor_final': valor_final,
        'ganancia': ganancia
    })
df_backtest = pd.DataFrame(resultados).sort_values('score_2022', ascending=False)
df_backtest.index = range(1, len(df_backtest) + 1)

print('✅ Setup completado')
print(f'   Precios: {precios.shape[0]} días, {precios.shape[1]} empresas')
print(f'   df_backtest: {len(df_backtest)} empresas PYMEs')
print(f'   df_2022: {len(df_2022)} empresas con score')

[*********************100%***********************]  3 of 3 completed

✅ Setup completado
   Precios: 500 días, 3 empresas
   df_backtest: 9 empresas PYMEs
   df_2022: 9 empresas con score


## 1. Grid search de parámetros con vectorbt

In [8]:
# --- BLOQUE 1: GRID SEARCH DE VENTANAS DE MEDIA MÓVIL ---

# Probar todas las combinaciones de ventana rápida y lenta
ventanas_rapidas = [10, 15, 20, 25, 30]
ventanas_lentas  = [40, 50, 60, 70, 80]

resultados_grid = []

for v_rap in ventanas_rapidas:
    for v_len in ventanas_lentas:
        if v_rap >= v_len:  # la rápida debe ser menor que la lenta
            continue

        ma_r = precios.rolling(v_rap).mean()
        ma_l = precios.rolling(v_len).mean()

        entradas_g = (ma_r > ma_l) & (ma_r.shift(1) <= ma_l.shift(1))
        salidas_g  = (ma_r < ma_l) & (ma_r.shift(1) >= ma_l.shift(1))

        pf = vbt.Portfolio.from_signals(
            close=precios, entries=entradas_g, exits=salidas_g,
            init_cash=10000, fees=0.001, freq='D'
        )

        sharpe_medio  = pf.sharpe_ratio().mean()
        retorno_medio = pf.total_return().mean() * 100

        resultados_grid.append({
            'ventana_rapida':    v_rap,
            'ventana_lenta':     v_len,
            'sharpe_medio':      round(sharpe_medio, 3),
            'retorno_medio_pct': round(retorno_medio, 1)
        })

df_grid = pd.DataFrame(resultados_grid).sort_values(
    'sharpe_medio', ascending=False)

print('=== RESULTADOS DEL GRID SEARCH ===')
print(df_grid.head(10).to_string())
print(f'\nMejores parámetros: MA{int(df_grid.iloc[0]["ventana_rapida"])} / MA{int(df_grid.iloc[0]["ventana_lenta"])}')
print(f'Sharpe medio: {df_grid.iloc[0]["sharpe_medio"]}')
print(f'Retorno medio: {df_grid.iloc[0]["retorno_medio_pct"]}%')

=== RESULTADOS DEL GRID SEARCH ===
    ventana_rapida  ventana_lenta  sharpe_medio  retorno_medio_pct
21              30             50         1.251               79.2
16              25             50         1.224               77.8
12              20             60         1.173               72.0
3               10             70         1.143               71.1
11              20             50         1.120               72.2
7               15             60         1.118               68.7
17              25             60         1.102               65.7
22              30             60         1.086               64.0
8               15             70         1.082               62.8
2               10             60         1.072               62.4

Mejores parámetros: MA30 / MA50
Sharpe medio: 1.251
Retorno medio: 79.2%


## 2. Separación train-test — validación honesta

In [9]:
# --- BLOQUE 2: SEPARACIÓN TRAIN-TEST ---

# Train: primer 70% de los datos  │  Test: último 30%
n_total  = len(precios)
n_train  = int(n_total * 0.70)

precios_train = precios.iloc[:n_train]
precios_test  = precios.iloc[n_train:]

print(f'Train: {precios_train.index[0].date()} → {precios_train.index[-1].date()} ({n_train} días)')
print(f'Test:  {precios_test.index[0].date()} → {precios_test.index[-1].date()} ({len(precios_test)} días)')

# --- MEJORES PARÁMETROS DEL GRID SEARCH ---
v_rap_opt = int(df_grid.iloc[0]['ventana_rapida'])
v_len_opt = int(df_grid.iloc[0]['ventana_lenta'])
print(f'\nParámetros óptimos del grid search: MA{v_rap_opt} / MA{v_len_opt}')

# --- APLICAR SOBRE EL SET DE TEST ---
ma_r_test = precios_test.rolling(v_rap_opt).mean()
ma_l_test = precios_test.rolling(v_len_opt).mean()

entradas_test = (ma_r_test > ma_l_test) & (ma_r_test.shift(1) <= ma_l_test.shift(1))
salidas_test  = (ma_r_test < ma_l_test) & (ma_r_test.shift(1) >= ma_l_test.shift(1))

pf_test = vbt.Portfolio.from_signals(
    close=precios_test, entries=entradas_test, exits=salidas_test,
    init_cash=10000, fees=0.001, freq='D'
)

# --- APLICAR SOBRE EL SET DE TRAIN ---
ma_r_train = precios_train.rolling(v_rap_opt).mean()
ma_l_train = precios_train.rolling(v_len_opt).mean()

entradas_train = (ma_r_train > ma_l_train) & (ma_r_train.shift(1) <= ma_l_train.shift(1))
salidas_train  = (ma_r_train < ma_l_train) & (ma_r_train.shift(1) >= ma_l_train.shift(1))

pf_train = vbt.Portfolio.from_signals(
    close=precios_train, entries=entradas_train, exits=salidas_train,
    init_cash=10000, fees=0.001, freq='D'
)

# --- COMPARAR TRAIN vs TEST ---
print('\n=== RESULTADOS TRAIN vs TEST ===')
for empresa in precios.columns:
    sharpe_train = pf_train.sharpe_ratio()[empresa]
    sharpe_test  = pf_test.sharpe_ratio()[empresa]
    ret_train    = pf_train.total_return()[empresa] * 100
    ret_test     = pf_test.total_return()[empresa] * 100
    degradacion  = ((sharpe_train - sharpe_test) / sharpe_train * 100
                    if sharpe_train > 0 else 0)
    print(f'\n{empresa}:')
    print(f'  Train — Sharpe: {sharpe_train:.2f} | Return: {ret_train:.1f}%')
    print(f'  Test  — Sharpe: {sharpe_test:.2f} | Return: {ret_test:.1f}%')
    print(f'  Degradación Sharpe: {degradacion:.0f}%'
          + (' ✓ aceptable' if degradacion < 40 else ' ✗ posible overfitting'))

Train: 2024-05-06 → 2025-09-26 (350 días)
Test:  2025-09-29 → 2026-05-04 (150 días)

Parámetros óptimos del grid search: MA30 / MA50

=== RESULTADOS TRAIN vs TEST ===

American:
  Train — Sharpe: 0.94 | Return: 34.5%
  Test  — Sharpe: inf | Return: 0.0%
  Degradación Sharpe: -inf% ✓ aceptable

Delta:
  Train — Sharpe: 0.77 | Return: 22.7%
  Test  — Sharpe: 0.07 | Return: 0.1%
  Degradación Sharpe: 91% ✗ posible overfitting

United:
  Train — Sharpe: 2.12 | Return: 118.4%
  Test  — Sharpe: 0.25 | Return: 1.3%
  Degradación Sharpe: 88% ✗ posible overfitting


## 3. Optimización de pesos del scoring de PYMEs

In [10]:
# --- BLOQUE 3: OPTIMIZACIÓN DE PESOS DEL SCORING DE PYMES ---

# --- GRID SEARCH DE PESOS ---
# Probar distintas combinaciones de peso margen / crecimiento
# Los pesos suman siempre 1.0

pesos_margen = [0.4, 0.5, 0.6, 0.7, 0.8]
resultados_pesos = []

for w_margen in pesos_margen:
    w_crecimiento = round(1 - w_margen, 1)

    # Recalcular score con estos pesos
    score_temp = (
        df_2022['pct_margen'] * w_margen +
        pd.Series([0.5] * len(df_2022),
                  index=df_2022.index) * w_crecimiento
    )

    # Clasificar en cuartiles
    cuartil_temp = pd.cut(score_temp,
                          bins=[0, 0.33, 0.66, 1.0],
                          labels=['LOW', 'MID', 'TOP'])

    # Calcular correlación con crecimiento real
    crecimiento_real_serie = pd.Series(
        {r['empresa']: r['crecimiento_real_pct'] for r in resultados}
    )
    corr_temp = score_temp.corr(crecimiento_real_serie)

    # Calcular ganancia simulada con estos pesos
    inversion_temp = {'TOP': 10000, 'MID': 5000, 'LOW': 2000}
    ganancia_temp = 0
    for i, empresa in enumerate(df_2022.index):
        c = str(cuartil_temp.iloc[i])
        inv = inversion_temp.get(c, 5000)
        crec = [r['crecimiento_real_pct']
                for r in resultados if r['empresa'] == empresa]
        if crec:
            ganancia_temp += inv * crec[0] / 100

    resultados_pesos.append({
        'peso_margen':       w_margen,
        'peso_crecimiento':  w_crecimiento,
        'ganancia_simulada': round(ganancia_temp, 0),
        'correlacion':       round(corr_temp if not pd.isna(corr_temp)
                                  else 0, 3)
    })

df_pesos = pd.DataFrame(resultados_pesos).sort_values(
    'ganancia_simulada', ascending=False)

print('=== OPTIMIZACIÓN DE PESOS DEL SCORING ===')
print(df_pesos.to_string())
print(f'\nMejores pesos: margen={df_pesos.iloc[0]["peso_margen"]}, '
      f'crecimiento={df_pesos.iloc[0]["peso_crecimiento"]}')
print(f'Ganancia simulada: {df_pesos.iloc[0]["ganancia_simulada"]:,.0f}€')
print(f'Correlación: {df_pesos.iloc[0]["correlacion"]}')

=== OPTIMIZACIÓN DE PESOS DEL SCORING ===
   peso_margen  peso_crecimiento  ganancia_simulada  correlacion
1          0.5               0.5             5655.0        0.568
3          0.7               0.3             5634.0        0.568
2          0.6               0.4             5634.0        0.568
4          0.8               0.2             5634.0        0.568
0          0.4               0.6             5055.0        0.568

Mejores pesos: margen=0.5, crecimiento=0.5
Ganancia simulada: 5,655€
Correlación: 0.568
